# Predicting Airline Satisfaction · EDA + LightGBM baseline

**Kaggle Playground Series S6E10** · [competition](https://www.kaggle.com/competitions/playground-series-s6e10) · [GitHub repo](https://github.com/aditbytes/kaggle-s6e10-airline-satisfaction)

**The task.** For each passenger in `test.csv`, predict the probability that they were **satisfied** with their flight. We get their trip details (age, travel class, distance, delays) and 13 service ratings from 0 to 5.

**The metric: ROC AUC.** AUC scores how well predictions *rank* passengers: it is the chance that a random satisfied passenger gets a higher score than a random unsatisfied one. 0.5 = coin flip, 1.0 = perfect. Only the **order** of predictions matters, so they don't need to be calibrated probabilities.

**Contents:** 1 Setup · 2 Data overview · 3 EDA · 4 Cross-validation · 5 LightGBM · 6 Feature importance · 7 Submission · 8 What I tried. Runs top to bottom on a CPU.

## 1 · Setup

Imports, the data folder, and the constants used everywhere. The notebook finds the data both on Kaggle (`/kaggle/input/...`) and locally inside the repo (`../data`). A fixed random seed makes every run reproducible.

In [ ]:
import json
import os
import time
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

T0 = time.time()
ON_KAGGLE = Path("/kaggle/input").exists()


def find_data_dir() -> Path:
    candidates = [Path("/kaggle/input/playground-series-s6e10"),
                  Path("/kaggle/input/competitions/playground-series-s6e10"),
                  Path("../data"), Path("data")]
    for c in candidates:
        if (c / "train.csv").exists():
            return c
    if ON_KAGGLE:  # fall back to any attached folder that looks like this competition
        for p in Path("/kaggle/input").glob("**/train.csv"):
            if (p.parent / "sample_submission.csv").exists():
                return p.parent
    raise FileNotFoundError("train.csv not found: attach the competition data (Kaggle) or download it into ../data")


DATA_DIR = find_data_dir()
OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path(".")

ID, TARGET = "id", "satisfaction"
SEED, N_FOLDS = 42, 5
CATEGORICAL = ["Gender", "Customer Type", "Type of Travel", "Class"]
SERVICE_RATINGS = [
    "Inflight wifi service", "Departure/Arrival time convenient", "Ease of Online booking",
    "Gate location", "Food and drink", "Online boarding", "Seat comfort", "Inflight entertainment",
    "On-board service", "Leg room service", "Baggage handling", "Checkin service", "Cleanliness",
]
DELAYS = ["Departure Delay in Minutes", "Arrival Delay in Minutes"]
NUMERIC = ["Age", "Flight Distance", *DELAYS]
FEATURES = NUMERIC + SERVICE_RATINGS + CATEGORICAL

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print(f"data: {DATA_DIR} | on Kaggle: {ON_KAGGLE} | CPUs: {os.cpu_count()} | lightgbm {lgb.__version__}")